In [1]:
import os
import tempfile
from pathlib import Path

# Lightning temporary storage
TEMP_ROOT = "/tmp/project_temp"

Path(TEMP_ROOT).mkdir(parents=True, exist_ok=True)

# Force temp usage
os.environ["TMPDIR"] = TEMP_ROOT
os.environ["TEMP"] = TEMP_ROOT
os.environ["TMP"] = TEMP_ROOT

tempfile.tempdir = TEMP_ROOT

print("Python temp:", tempfile.gettempdir())

Python temp: /tmp/project_temp


In [2]:
import duckdb
import pandas as pd
import re
import json

In [ ]:
duck_temp = "/tmp/duckdb_spill"
os.makedirs(duck_temp, exist_ok=True)

con = duckdb.connect()

con.execute(f"""
SET temp_directory='{duck_temp}'
""")

# Optional but recommended
con.execute("""
SET memory_limit='7GB'
""")

print(
    con.execute(
        "SELECT current_setting('temp_directory')"
    ).fetchone()[0]
)

base_path = os.path.expanduser('~/secure_data')

data_path = f"{base_path}/**/*.parquet"

print(data_path)


/tmp/duckdb_spill
/teamspace/studios/this_studio/secure_data/**/*.parquet


##### Delete Missing Values in Cricial Columns , Zero Variance Columns and Delete Gift Bundles

In [4]:

columns_to_exclude = "usage_type, volume, call_duration, dynamic_speed, gender, date_of_birth"


print(" [1/3] Inspecting data and generating detailed exclusion report...\n")

diagnostic_query = f"""
SELECT 
    COUNT(*) AS "1. Total Original Rows",
    SUM(CASE WHEN msisdn IS NULL THEN 1 ELSE 0 END) AS "2. Missing: Phone Number (msisdn)",
    SUM(CASE WHEN tbl_dt IS NULL THEN 1 ELSE 0 END) AS "3. Missing: Date (tbl_dt)",
    SUM(CASE WHEN bundle_id IS NULL THEN 1 ELSE 0 END) AS "4. Missing: Bundle ID (bundle_id)",
    SUM(CASE WHEN bundle_name IS NULL THEN 1 ELSE 0 END) AS "5. Missing: Bundle Name (bundle_name)",
    SUM(CASE WHEN bundle_type IS NULL THEN 1 ELSE 0 END) AS "6. Missing: Bundle Type (bundle_type)",
    SUM(CASE WHEN COALESCE(product_category, '') = 'STAFF' THEN 1 ELSE 0 END) AS "7. Excluded: Staff Bundles (STAFF)",
    SUM(CASE WHEN COALESCE(product_category, '') = 'VAS' THEN 1 ELSE 0 END) AS "8. Excluded: Value Added Services (VAS)",
    SUM(CASE WHEN COALESCE(product_category, '') = 'DIY' OR bundle_name ILIKE '%DIY%' THEN 1 ELSE 0 END) AS "9. Excluded: Custom Bundles (DIY)",
    SUM(CASE WHEN COALESCE(canal, '') = 'KDO' THEN 1 ELSE 0 END) AS "10. Excluded: Gift Bundles (KDO)"
FROM read_parquet('{data_path}')
"""


report_df = con.sql(diagnostic_query).df().T
report_df.columns = ["Rows Matching Condition"]
display(report_df)


removed_kdo_count = report_df.iloc[9, 0]


print("\n [2/3] Applying filters and building the final clean table...")

preprocess_query = f"""
CREATE OR REPLACE TABLE clean_final_data AS 
SELECT DISTINCT * EXCLUDE ({columns_to_exclude})
FROM read_parquet('{data_path}')
WHERE 
    
    msisdn IS NOT NULL 
    AND tbl_dt IS NOT NULL
    AND bundle_id IS NOT NULL 
    AND bundle_name IS NOT NULL
    AND bundle_type IS NOT NULL
    
   
    AND COALESCE(product_category, '') NOT IN ('DIY', 'STAFF', 'VAS')
    AND bundle_name NOT ILIKE '%DIY%'
    AND COALESCE(canal, '') != 'KDO'
"""

con.sql(preprocess_query)


initial_count = report_df.iloc[0, 0]
final_count = con.sql("SELECT COUNT(*) FROM clean_final_data").fetchone()[0]
total_removed = initial_count - final_count

print(f"\n Process completed successfully!")
print(f"▪️ Initial Size: {initial_count:,} rows.")
print(f"▪️ Gift bundles (KDO) identified & removed: {removed_kdo_count:,} rows.")
print(f"▪️ Final Clean Table Size: {final_count:,} rows.")
print(f"▪️ Total records filtered out (including exact duplicates): {total_removed:,} rows.")
print("\nTable 'clean_final_data' is now ready and optimized!")

 [1/3] Inspecting data and generating detailed exclusion report...



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Rows Matching Condition
1. Total Original Rows,159801139.0
2. Missing: Phone Number (msisdn),0.0
3. Missing: Date (tbl_dt),0.0
4. Missing: Bundle ID (bundle_id),19168.0
5. Missing: Bundle Name (bundle_name),19404.0
6. Missing: Bundle Type (bundle_type),19404.0
7. Excluded: Staff Bundles (STAFF),0.0
8. Excluded: Value Added Services (VAS),10765737.0
9. Excluded: Custom Bundles (DIY),31217138.0
10. Excluded: Gift Bundles (KDO),38033261.0



 [2/3] Applying filters and building the final clean table...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


 Process completed successfully!
▪️ Initial Size: 159,801,139.0 rows.
▪️ Gift bundles (KDO) identified & removed: 38,033,261.0 rows.
▪️ Final Clean Table Size: 104,162,559 rows.
▪️ Total records filtered out (including exact duplicates): 55,638,580.0 rows.

Table 'clean_final_data' is now ready and optimized!


##### Cleaning Price Column

In [ ]:
print("Processing price and keeping rows for inspection...")

clean_price_query = """
CREATE OR REPLACE TABLE clean_final_data AS 
WITH RegexCleaned AS (
    SELECT 
        *,
        price AS original_price, 
        
        regexp_replace(CAST(price AS VARCHAR), '[^0-9.,]', '', 'g') AS price_numeric_chars,
        
        regexp_extract(bundle_name, '@([0-9]+)', 1) AS price_from_bundle
    FROM clean_final_data
),
FormattedPrice AS (
    SELECT 
        *,
        TRY_CAST(
            CASE 
                
                WHEN price_numeric_chars LIKE '%,%.%' THEN REPLACE(price_numeric_chars, ',', '')
                WHEN price_numeric_chars LIKE '%,%' THEN REPLACE(price_numeric_chars, ',', '.')
                WHEN TRY_CAST(price_numeric_chars AS DOUBLE) IS NOT NULL THEN price_numeric_chars
                
                
                WHEN price_from_bundle IS NOT NULL THEN price_from_bundle
                
                ELSE NULL
            END AS DOUBLE
        ) AS price_clean
    FROM RegexCleaned
)

SELECT * EXCLUDE (price, price_numeric_chars, price_from_bundle), price_clean AS price
FROM FormattedPrice;
"""

con.sql(clean_price_query)


failed_count_query = "SELECT COUNT(*) FROM clean_final_data WHERE price IS NULL"
failed_count = con.sql(failed_count_query).fetchone()[0]

print(f"Number of rows where price conversion failed: {failed_count:,} rows.")

if failed_count > 0:
    print("\nSample of values that STILL caused failure (after checking bundle name):")
    failed_sample_query = """
    SELECT 
        original_price AS "Original Price", 
        bundle_name AS "Bundle Name"
    FROM clean_final_data 
    WHERE price IS NULL
    LIMIT 15;
    """
    display(con.sql(failed_sample_query).df())
    
    print("\nDeleting the failed rows...")
    con.sql("DELETE FROM clean_final_data WHERE price IS NULL;")
    print(f"Successfully deleted {failed_count:,} rows.")
else:
    print("Excellent! All prices were converted successfully (either from price column or bundle name).")


con.sql("ALTER TABLE clean_final_data DROP COLUMN original_price;")
print("\nOld column dropped successfully. The table now only contains the processed column under the name 'price'!")


final_size = con.sql("SELECT COUNT(*) FROM clean_final_data").fetchone()[0]
print(f"Final table size after deletion: {final_size:,} rows.")

Processing price and keeping rows for inspection...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Number of rows where price conversion failed: 392 rows.

Sample of values that STILL caused failure (after checking bundle name):


,Original Price,Bundle Name
0,F,SMS Bundle @F
1,F,SMS Bundle @F
2,F,SMS Bundle @F
3,F,SMS Bundle @F
4,F,SMS Bundle @F
5,F,SMS Bundle @F
6,F,SMS Bundle @F
7,F,SMS Bundle @F
8,F,SMS Bundle @F
9,F,SMS Bundle @F



Deleting the failed rows...
Successfully deleted 392 rows.

Old column dropped successfully. The table now only contains the processed column under the name 'price'!
Final table size after deletion: 104,162,167 rows.


##### Cleaning Validity Column

In [6]:


print("Processing the 'Validity' column and converting it to hours...")


validity_query = """
CREATE OR REPLACE TABLE clean_final_data AS 
WITH ExtractedData AS (
    SELECT 
        *,
        -- Extract the first number found in the text (integer or decimal)
        TRY_CAST(REGEXP_EXTRACT(LOWER(validity), '[0-9]+(\\.[0-9]+)?') AS DOUBLE) AS extracted_number,
        LOWER(CAST(validity AS VARCHAR)) AS val_text
    FROM clean_final_data
)
SELECT 
    * EXCLUDE (extracted_number, val_text),
    CASE 
        WHEN val_text LIKE '%month%' THEN extracted_number * 30.0 * 24.0
        WHEN val_text LIKE '%day%' THEN extracted_number * 24.0
        WHEN val_text LIKE '%hour%' THEN extracted_number
        ELSE extracted_number -- Default logic: return the number as is
    END AS validity_hours
FROM ExtractedData;
"""

con.sql(validity_query)


null_count = con.sql("SELECT COUNT(*) FROM clean_final_data WHERE validity_hours IS NULL").fetchone()[0]

if null_count > 0:
    print(f"Warning: Found {null_count:,} records where validity extraction failed (resulted in NULL).")
    print("Sample of values causing the issue:")
    display(con.sql("SELECT validity FROM clean_final_data WHERE validity IS NOT NULL AND bundle_id IN (SELECT bundle_id FROM clean_final_data WHERE validity_hours IS NULL) LIMIT 5").df())
else:
    print("Processing completed successfully with no missing values.")


# Drop the old 'validity' column and rename 'validity_hours' to the original name
con.sql("ALTER TABLE clean_final_data DROP COLUMN validity")
con.sql("ALTER TABLE clean_final_data RENAME COLUMN validity_hours TO validity")

print("Old column removed and replaced with the processed validity values.")

# Display sample results
display(con.sql("SELECT bundle_name, validity FROM clean_final_data LIMIT 5").df())

Processing the 'Validity' column and converting it to hours...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Processing completed successfully with no missing values.
Old column removed and replaced with the processed validity values.


,bundle_name,validity
0,Forfait Maxivoice 10Mins 1 jour@150F,24.0
1,Forfait Maxivoice 7Mins 1 jour@120F,24.0
2,SMS Bundle 3 Days@122F,72.0
3,Forfait Maxivoice 16Mins 1 jour@200F,24.0
4,SMS Bundle 1 Day @26F,24.0


#####  Advanced Volume Extraction & Feature Recovery

In [9]:

print("Initializing volume extraction functions and integrating into DuckDB (via UDF)...")

# ==========================================
# 1. Dictionaries and Pattern Definitions
# ==========================================
TYPE_MAP = {
    'BUNDLE_VOICE': 'VOICE',
    'BUNDLE_SMS':   'SMS',
    'BUNDLE_DATA':  'DATA',
}

patterns = {
    'DATA': [r'(\d[\d,.]*)\s*gb', r'(\d[\d,.]*)\s*go', r'(\d[\d,.]*)\s*mb', r'(\d[\d,.]*)\s*mo'],
    'VOICE': [r'(\d[\d,.]*)\s*mn', r'(\d[\d,.]*)\s*min', r'(\d[\d,.]*)\s*mins', r'(\d[\d,.]*)\s*appel', r'(\d[\d,.]*)\s*voc'],
    'SMS': [r'(\d[\d,.]*)\s*sms', r'(\d[\d,.]*)\s*msg'],
}

MIXED_ORDER = ['DATA', 'VOICE', 'SMS']

unlimited_patterns = [
    r'illimit', r'unlimit', r'illimité', r'unlimité', r'illimitée', r'unlimitée', r'infini',
]

UNIT_MAP = {
    'DATA': {
        r'(\d[\d,.]*)\s*gb': 'GB', r'(\d[\d,.]*)\s*go': 'GB',
        r'(\d[\d,.]*)\s*mb': 'MB', r'(\d[\d,.]*)\s*mo': 'MB',
    },
    'VOICE': {
        r'(\d[\d,.]*)\s*mn': 'min', r'(\d[\d,.]*)\s*min': 'min', r'(\d[\d,.]*)\s*mins': 'min',
        r'(\d[\d,.]*)\s*appel': 'appel', r'(\d[\d,.]*)\s*voc': 'voc',
    },
    'SMS': {
        r'(\d[\d,.]*)\s*sms': 'SMS', r'(\d[\d,.]*)\s*msg': 'msg',
    },
}

_VOICE_UNITS_RE = re.compile(r'(\d[\d,.]*)\s*(mins?|mn|sec|s)', re.IGNORECASE)

# ==========================================
# 2. Helper Functions
# ==========================================
def _parse_complex_voice(text):
    if not text: return None
    total_minutes = 0.0
    matched = False
    
    slash_match = re.search(r'(\d[\d,.]*)\s*/\s*(\d[\d,.]*)\s*(mins?|mn)', text, re.IGNORECASE)
    if slash_match:
        val1 = float(slash_match.group(1).replace(',', '.'))
        val2 = float(slash_match.group(2).replace(',', '.'))
        total_minutes += (val1 + val2)
        matched = True
        text = text.replace(slash_match.group(0), '')

    hits = _VOICE_UNITS_RE.findall(text)
    if hits:
        matched = True
        for val, unit in hits:
            num = float(val.replace(',', '.'))
            if unit.lower() in ['sec', 's']:
                total_minutes += num / 60  
            else:
                total_minutes += num

    if not matched: return None
    return f"{int(total_minutes)}min" if total_minutes == int(total_minutes) else f"{round(total_minutes, 2)}min"

def _search_type(name_lower, btype):
    for pat in patterns.get(btype, []):
        m = re.search(pat, name_lower, re.IGNORECASE)
        if m:
            raw = m.group(1).replace(',', '.')
            unit = UNIT_MAP.get(btype, {}).get(pat, '')
            return f"{raw}{unit}"
    return None

def extract_volume_from_name(bundle_name, bundle_type):
    if not bundle_name: return None
    name_lower = str(bundle_name).lower()
    raw_type   = str(bundle_type).upper().strip() if bundle_type else ''
    btype      = TYPE_MAP.get(raw_type, raw_type)  

    for pat in unlimited_patterns:
        if re.search(pat, name_lower, re.IGNORECASE):
            return 'Unlimited'

    if btype in ('VOICE', 'MIXED'):
        complex_result = _parse_complex_voice(bundle_name)
        if complex_result: return complex_result

    if btype == 'MIXED':
        for t in MIXED_ORDER:
            result = _search_type(name_lower, t)
            if result: return result
        return None

    return _search_type(name_lower, btype)

# ==========================================
# 3. Registering Function in DuckDB
# ==========================================
# Remove old function if it exists
try:
    con.remove_function('extract_volume_udf')
except Exception:
    pass 

# Register the new function with null_handling='SPECIAL'
con.create_function(
    'extract_volume_udf', 
    extract_volume_from_name, 
    ['VARCHAR', 'VARCHAR'], 
    'VARCHAR', 
    null_handling='SPECIAL'
)

# ==========================================
# 4. Applying Extraction to Data
# ==========================================
missing_before = con.sql("SELECT COUNT(*) FROM clean_final_data WHERE configured_volume IS NULL").fetchone()[0]
print(f"Missing volume values before processing: {missing_before:,}")

# query = """
# CREATE OR REPLACE TABLE clean_final_data AS
# SELECT
#     * EXCLUDE (configured_volume),
#     -- If volume is missing, call the UDF to extract it from the name
#     COALESCE(CAST(configured_volume AS VARCHAR), extract_volume_udf(CAST(bundle_name AS VARCHAR), CAST(bundle_type AS VARCHAR))) AS configured_volume
# FROM clean_final_data;
# """
# con.sql(query)

query = """
CREATE OR REPLACE TABLE clean_final_data AS
SELECT
    *,
    configured_volume AS old_configured_volume, 
    
    COALESCE(CAST(configured_volume AS VARCHAR), extract_volume_udf(CAST(bundle_name AS VARCHAR), CAST(bundle_type AS VARCHAR))) AS new_configured_volume
FROM clean_final_data;
"""
con.sql(query)

# 2. ترتيب الأعمدة: حذف القديم وتغيير اسم الجديد ليكون هو الأساسي
con.sql("ALTER TABLE clean_final_data DROP COLUMN configured_volume")
con.sql("ALTER TABLE clean_final_data RENAME COLUMN new_configured_volume TO configured_volume")



# ==========================================
# 5. Printing Results and Verification
# ==========================================
missing_after = con.sql("SELECT COUNT(*) FROM clean_final_data WHERE configured_volume IS NULL").fetchone()[0]
filled = missing_before - missing_after

print(f"\nProcessing completed successfully!")
print(f"Values successfully extracted and filled: {filled:,}")
print(f"Values still missing (not found): {missing_after:,}")

if filled > 0:
    print("\nSample of records where volume was extracted from the name (Verification):")
    sample_query = """
    SELECT 
        bundle_name, 
        bundle_type, 
        configured_volume AS "Extracted_Volume"
    FROM clean_final_data 
    WHERE old_configured_volume IS NULL AND configured_volume IS NOT NULL
    LIMIT 10;
    """
    display(con.sql(sample_query).df())
    
    # sample_query = """
    # SELECT v.bundle_name, v.bundle_type, v.configured_volume AS "Extracted_Volume"
    # FROM clean_final_data v
    # JOIN clean_final_data old ON v.bundle_name = old.bundle_name
    # WHERE old.configured_volume IS NULL AND v.configured_volume IS NOT NULL
    # LIMIT 10;
    # """
    # display(con.sql(sample_query).df())


con.sql("ALTER TABLE clean_final_data DROP COLUMN old_configured_volume")

Initializing volume extraction functions and integrating into DuckDB (via UDF)...


Missing volume values before processing: 23,702,356


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Processing completed successfully!
Values successfully extracted and filled: 1,608,600
Values still missing (not found): 22,093,756

Sample of records where volume was extracted from the name (Verification):


,bundle_name,bundle_type,Extracted_Volume
0,Forfait tous reseaux 2 mins 1 jour@100F + BONUS,BUNDLE_VOICE,2min
1,Forfait tous reseaux 2 mins 1 jour@100F + BONUS,BUNDLE_VOICE,2min
2,Forfait tous reseaux 2 mins 1 jour@100F + BONUS,BUNDLE_VOICE,2min
3,Forfait tous reseaux 2 mins 1 jour@100F + BONUS,BUNDLE_VOICE,2min
4,Forfait tous reseaux 3 mins 1 jour@150F + BONUS,BUNDLE_VOICE,3min
5,Forfait tous reseaux 2 mins 1 jour@100F + BONUS,BUNDLE_VOICE,2min
6,Forfait tous reseaux 2 mins 1 jour@100F + BONUS,BUNDLE_VOICE,2min
7,Forfait tous reseaux 2 mins 1 jour@100F + BONUS,BUNDLE_VOICE,2min
8,Forfait tous reseaux 5 mins 1 jour@250F + BONUS,BUNDLE_VOICE,5min
9,Forfait appels 5 mins 1 jour@25F,BUNDLE_VOICE,5min


In [10]:
print("Analyzing bundle names where volume extraction failed...")

diagnostic_query = """
WITH MissingVolumes AS (
    -- Get all rows where the volume is still missing
    SELECT bundle_name, bundle_type
    FROM clean_final_data
    WHERE configured_volume IS NULL
),
TotalMissing AS (
    -- Calculate the grand total of missing records (for percentage calculation)
    SELECT COUNT(*) as total_count FROM MissingVolumes
)
SELECT 
    m.bundle_name AS "Bundle Name (Issue)",
    m.bundle_type AS "Bundle Type",
    COUNT(*) AS "Frequency (Row Count)",
    ROUND(COUNT(*) * 100.0 / MAX(t.total_count), 2) AS "Percentage of Total Missing %"
FROM MissingVolumes m
CROSS JOIN TotalMissing t
GROUP BY m.bundle_name, m.bundle_type
ORDER BY "Frequency (Row Count)" DESC
LIMIT 20;
"""

display(con.sql(diagnostic_query).df())

Analyzing bundle names where volume extraction failed...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Bundle Name (Issue),Bundle Type,Frequency (Row Count),Percentage of Total Missing %
0,SMS Bundle 1 Day @26F,BUNDLE_SMS,7428808,33.62
1,SMS Bundle 2 Days @66F,BUNDLE_SMS,6836804,30.94
2,SMS Bundle 1 Day @41F,BUNDLE_SMS,4109466,18.60
3,SMS Bundle 3 Days@122F,BUNDLE_SMS,3608126,16.33
4,SMS Bundle 5 Days@335F,BUNDLE_SMS,86408,0.39
5,SMS Bundle 10 Days@507F,BUNDLE_SMS,16646,0.08
6,SMS Bundle 1 day@50F,BUNDLE_SMS,2540,0.01
7,SMS Bundle 3 Days@100F,BUNDLE_SMS,1317,0.01
8,Flexi Bundle 2 Days @150F,FLEXI_BUNDLE,1145,0.01
9,Whatsapp text 1 jour@10F,BUNDLE_DATA,694,0.00


In [11]:
print("Analyzing missing volume bundle frequencies...")

# ==========================================
# 1. Create a temporary table to calculate frequency
# ==========================================
freq_query = """
CREATE OR REPLACE TEMP TABLE MissingFreq AS
SELECT bundle_name, COUNT(*) as freq
FROM clean_final_data
WHERE configured_volume IS NULL
GROUP BY bundle_name;
"""
con.sql(freq_query)

# ==========================================
# 2. Display a sample of data to be deleted (<= 432)
# ==========================================
print("\nSample of rare bundles to be deleted (Frequency < 1317):")
sample_delete_query = """
SELECT DISTINCT v.bundle_name AS "Bundle Name to Delete", f.freq AS "Frequency"
FROM clean_final_data v
JOIN MissingFreq f ON v.bundle_name = f.bundle_name
WHERE v.configured_volume IS NULL AND f.freq < 1317
ORDER BY f.freq DESC
LIMIT 15;
"""
display(con.sql(sample_delete_query).df())

# ==========================================
# 3. Execution (Deletion and Imputation)
# ==========================================
print("\nApplying filters (Deleting noise and filling Unlimited)...")

# Delete records with frequency of 1317 or less
con.sql("""
DELETE FROM clean_final_data
WHERE configured_volume IS NULL 
AND bundle_name IN (SELECT bundle_name FROM MissingFreq WHERE freq < 1317);
""")

# Fill the remaining high-frequency NULLs with 'Unlimited'
con.sql("""
UPDATE clean_final_data 
SET configured_volume = 'Unlimited' 
WHERE configured_volume IS NULL;
""")

# ==========================================
# 4. Final Verification
# ==========================================
final_missing = con.sql("SELECT COUNT(*) FROM clean_final_data WHERE configured_volume IS NULL").fetchone()[0]

print(f"Process completed successfully!")
print(f"Current missing values count: {final_missing}")
print("Noise removed. High-frequency time-based bundles imputed as 'Unlimited'.")

Analyzing missing volume bundle frequencies...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Sample of rare bundles to be deleted (Frequency < 1317):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Bundle Name to Delete,Frequency
0,Flexi Bundle 2 Days @150F,1145
1,Whatsapp text 1 jour@10F,694
2,Whatsapp text 1 jour@50F,647
3,Flexi Bundle 2 Days @200F,432
4,Flexi Bundle 2 Days @120F,237
5,SMS Bundle 1 day@15F,195
6,Flexi Bundle 2 Days @220F,66
7,PRO GFU 10-23750F,51
8,PRO GFU 5-12000F,38
9,Flexi Bundle 7 Days @420F,34



Applying filters (Deleting noise and filling Unlimited)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Process completed successfully!
Current missing values count: 0
Noise removed. High-frequency time-based bundles imputed as 'Unlimited'.


In [12]:



print("Initializing advanced volume disentanglement function...")

# ==========================================
# 1. Dictionaries and Patterns
# ==========================================
_NAME_DATA_PATTERNS = [
    (r'(\d[\d,.]*)\s*gb', 'GB'),
    (r'(\d[\d,.]*)\s*go', 'GB'),
    (r'(\d[\d,.]*)\s*mb', 'MB'),
    (r'(\d[\d,.]*)\s*mo', 'MB'),
]

TYPE_MAP = {
    'BUNDLE_VOICE': 'VOICE',
    'BUNDLE_SMS':   'SMS',
    'BUNDLE_DATA':  'DATA',
}

_VOICE_COMPOUND_RE = re.compile(
    r'(\d[\d,.]*)\s*(?:mins?|mn)[^,]*?(\d[\d,.]*)\s*(?:sec|s)\b',
    re.IGNORECASE
)

def _unit_from_name(bundle_name):
    """Infers the unit from the bundle name if not explicitly stated"""
    name_lower = str(bundle_name).lower()
    for pat, unit in _NAME_DATA_PATTERNS:
        if re.search(pat, name_lower, re.IGNORECASE):
            return unit
    return 'MB'  

# ==========================================
# 2. Updated Processing Function (returns JSON instead of DataFrame)
# ==========================================
def parse_volumes_to_json(volume_str, bundle_type, bundle_name):
    """
    Parses volume and returns a JSON string: {"MB": x, "MIN": y, "SMS": z}
    Replaces Unlimited with -1.0 in the specific orthogonal column.
    """
    vol_MB, vol_min, vol_SMS = 0.0, 0.0, 0.0
    
    if not volume_str or str(volume_str).strip().lower() in ('nan', 'none', ''):
        return json.dumps({"MB": vol_MB, "MIN": vol_min, "SMS": vol_SMS})

    v = str(volume_str).strip().lower()
    btype = TYPE_MAP.get(str(bundle_type).upper().strip(), str(bundle_type).upper().strip())
    name_lower = str(bundle_name).lower() if bundle_name else ''



    if v in ('unlimited', 'illimité', 'illimitée', 'unlimitée', 'infini', '-1', '-1.0'):
        if btype == 'VOICE': vol_min = -1.0
        elif btype == 'SMS': vol_SMS = -1.0
        elif btype == 'DATA': vol_MB = -1.0
        else:
            # Try to infer from name if type is unclear
            if 'sms' in name_lower: vol_SMS = -1.0
            elif 'voix' in name_lower or 'min' in name_lower: vol_min = -1.0
            else: vol_MB = -1.0
            
        return json.dumps({"MB": vol_MB, "MIN": vol_min, "SMS": vol_SMS})

    # Handling compound minutes and seconds (Compound Voice)
    compound = _VOICE_COMPOUND_RE.search(v)
    if compound:
        mins = float(compound.group(1).replace(',', '.'))
        secs = float(compound.group(2).replace(',', '.'))
        vol_min = round(mins + secs / 60, 4)
        return json.dumps({"MB": vol_MB, "MIN": vol_min, "SMS": vol_SMS})

    # Regular extraction
    m = re.search(r'(\d[\d,.]*)\s*([a-zA-Z]*)', v.replace('\xa0', ' '))
    if not m:
        return json.dumps({"MB": vol_MB, "MIN": vol_min, "SMS": vol_SMS})

    num_str = m.group(1).replace(' ', '').replace(',', '.')
    unit_str = m.group(2).strip().lower()

    try:
        num = float(num_str)
    except ValueError:
        return json.dumps({"MB": vol_MB, "MIN": vol_min, "SMS": vol_SMS})

    # Unit conversion and assignment to the correct column
    actual_unit = unit_str.upper()
    if actual_unit in ('GB', 'GO', 'G'):
        vol_MB = round(num * 1024, 4)
    elif actual_unit in ('MB', 'MO'):
        vol_MB = num
    elif actual_unit in ('MIN', 'MINS', 'MN'):
        vol_min = num
    elif actual_unit == 'SMS':
        vol_SMS = int(num)
    elif actual_unit == '':
        # If unit is missing, rely on bundle type
        if btype == 'VOICE': vol_min = num
        elif btype == 'SMS': vol_SMS = int(num)
        else: 
            inferred = _unit_from_name(bundle_name)
            if inferred == 'GB': vol_MB = round(num * 1024, 4)
            elif inferred == 'MB': vol_MB = num
            else: vol_MB = num
    else:
        vol_MB = num # Default

    return json.dumps({"MB": vol_MB, "MIN": vol_min, "SMS": vol_SMS})

# ==========================================
# 3. Integrate Function into DuckDB and Build Columns
# ==========================================
try:
    con.remove_function('parse_volumes_udf')
except Exception:
    pass 

# Register function with Null handling settings
con.create_function(
    'parse_volumes_udf', 
    parse_volumes_to_json, 
    ['VARCHAR', 'VARCHAR', 'VARCHAR'], 
    'VARCHAR', 
    null_handling='SPECIAL'
)

print("Applying function and separating the three columns...")

query = """
CREATE OR REPLACE TABLE clean_final_data AS 
WITH ParsedJSON AS (
    SELECT 
        *,
        -- Apply Python function returning JSON format
        parse_volumes_udf(
            CAST(configured_volume AS VARCHAR), 
            CAST(bundle_type AS VARCHAR), 
            CAST(bundle_name AS VARCHAR)
        ) AS volume_json
    FROM clean_final_data
)
SELECT 
    * EXCLUDE (configured_volume, volume_json),
    -- Extract values from JSON and cast them to DOUBLE
    CAST(json_extract_string(volume_json, '$.MB') AS DOUBLE) AS data_volume_mb,
    CAST(json_extract_string(volume_json, '$.MIN') AS DOUBLE) AS voice_volume_min,
    CAST(json_extract_string(volume_json, '$.SMS') AS DOUBLE) AS sms_volume_count
FROM ParsedJSON;
"""
con.sql(query)

print("Process completed successfully! The configured_volume column was dropped and replaced with 3 orthogonal columns.")

# ==========================================
# 4. Display a Sample to Verify Value Separation and Unlimited (-1)
# ==========================================
check_query = """
SELECT 
    bundle_name, 
    bundle_type, 
    data_volume_mb AS "Data (MB)", 
    voice_volume_min AS "Voice (Min)", 
    sms_volume_count AS "SMS"
FROM clean_final_data 
WHERE data_volume_mb = -1 OR voice_volume_min = -1 OR sms_volume_count = -1
   OR data_volume_mb > 0 OR voice_volume_min > 0 OR sms_volume_count > 0
LIMIT 15;
"""
display(con.sql(check_query).df())

Initializing advanced volume disentanglement function...
Applying function and separating the three columns...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Process completed successfully! The configured_volume column was dropped and replaced with 3 orthogonal columns.


,bundle_name,bundle_type,Data (MB),Voice (Min),SMS
0,Forfait Maxivoice 10Mins 1 jour@150F,BUNDLE_VOICE,0.0,10.0,0.0
1,Forfait Maxivoice 7Mins 1 jour@120F,BUNDLE_VOICE,0.0,7.0,0.0
2,SMS Bundle 3 Days@122F,BUNDLE_SMS,0.0,0.0,-1.0
3,Forfait Maxivoice 16Mins 1 jour@200F,BUNDLE_VOICE,0.0,16.0,0.0
4,SMS Bundle 1 Day @26F,BUNDLE_SMS,0.0,0.0,-1.0
5,NDEKO Net 220MB 1 jour@281F,BUNDLE_DATA,220.0,0.0,0.0
6,Forfait Maxivoice 5Mins 1 jour@100F,BUNDLE_VOICE,0.0,5.0,0.0
7,Forfait Maxivoice 5Mins 1 jour@100F,BUNDLE_VOICE,0.0,5.0,0.0
8,SMS Bundle 2 Days @66F,BUNDLE_SMS,0.0,0.0,-1.0
9,Maxivoice 43Mins jour@525F,BUNDLE_VOICE,0.0,43.0,0.0


In [13]:

remaining_columns_df = con.sql("DESCRIBE clean_final_data").df()


print(f"Number of columns remaining: {len(remaining_columns_df)}\n")

print("List of remaining columns:")
for i, col in enumerate(remaining_columns_df['column_name'], 1):
    print(f"{i}. {col}")

Number of columns remaining: 30

List of remaining columns:
1. tbl_dt
2. msisdn
3. bundle_id
4. bundle_name
5. bundle_type
6. subscriptions
7. total_rev
8. category_description
9. service_class_category
10. product_category
11. canal
12. payment_mode
13. business_categorisation
14. cell_id
15. cell_name
16. site_id
17. site_name
18. latitude
19. longitude
20. department_city
21. technology
22. model_name
23. brand_name
24. device_capability
25. price_clean
26. price
27. validity
28. data_volume_mb
29. voice_volume_min
30. sms_volume_count


##### Feature Purification

In [15]:


print("Executing final column filtering and building the model table (Purified Dataset)...")

# ==========================================
# 1. Create the Final Purified Table (12 Columns)
# ==========================================
filter_columns_query = """
CREATE OR REPLACE TABLE final_model_data AS 
SELECT 
    -- Group 1: Core Sequence Columns
    msisdn,
    bundle_id,
    tbl_dt,
    
    -- Group 2: Catalog Metadata & Business Rules
    bundle_name,
    bundle_type,
    product_category,
    service_class_category, -- Added as a business constraint for post-recommendation filtering
    
    -- Group 3: Orthogonal Numeric Features
    price,
    validity,
    data_volume_mb,
    voice_volume_min,
    sms_volume_count

FROM clean_final_data;
"""

con.sql(filter_columns_query)

# ==========================================
# 2. Calculate and Display Filtering Results
# ==========================================
initial_cols = len(con.sql("DESCRIBE clean_final_data").df())
final_cols = len(con.sql("DESCRIBE final_model_data").df())
reduction_pct = ((initial_cols - final_cols) / initial_cols) * 100

print(f"Filtering process completed successfully!")
print(f"Columns reduced from {initial_cols} to {final_cols} (Reduction Ratio: {reduction_pct:.1f}%).")
print(f"Table 'final_model_data' is now ready for the K-Core filtering stage.")

# ==========================================
# 3. Display Final Table Schema for Verification
# ==========================================
print("\nFinal Table Structure:")
display(con.sql("DESCRIBE final_model_data").df()[['column_name', 'column_type']])

# Note: If you want to free up memory by permanently deleting the old table (30 columns),
# you can uncomment and execute the following line:
# con.sql("DROP TABLE clean_final_data;")

Executing final column filtering and building the model table (Purified Dataset)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Filtering process completed successfully!
Columns reduced from 30 to 12 (Reduction Ratio: 60.0%).
Table 'final_model_data' is now ready for the K-Core filtering stage.

Final Table Structure:


,column_name,column_type
0,msisdn,BIGINT
1,bundle_id,VARCHAR
2,tbl_dt,BIGINT
3,bundle_name,VARCHAR
4,bundle_type,VARCHAR
5,product_category,VARCHAR
6,service_class_category,VARCHAR
7,price,DOUBLE
8,validity,DOUBLE
9,data_volume_mb,DOUBLE


In [16]:

print(" Running Data Profiling... This may take about a minute due to the large volume of data.\n")


summary_query = f"SUMMARIZE SELECT * FROM final_model_data"
df_summary = con.sql(summary_query).df()


result_df = df_summary[['column_name', 'column_type', 'null_percentage', 'approx_unique']].copy()


total_rows = df_summary['count'].max()
result_df['null_count'] = (result_df['null_percentage'] / 100 * total_rows).astype(int)


result_df = result_df[['column_name', 'column_type', 'null_count', 'null_percentage', 'approx_unique']]
result_df.rename(columns={
    'column_name': 'Column Name',
    'column_type': 'Data Type',
    'null_count': 'Null Count',
    'null_percentage': 'Null Percentage %',
    'approx_unique': 'Approx Unique Values'
}, inplace=True)


pd.set_option('display.max_rows', None)
display(result_df)
pd.reset_option('display.max_rows')

 Running Data Profiling... This may take about a minute due to the large volume of data.



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Column Name,Data Type,Null Count,Null Percentage %,Approx Unique Values
0,msisdn,BIGINT,0,0.0,2364334
1,bundle_id,VARCHAR,0,0.0,970
2,tbl_dt,BIGINT,0,0.0,123
3,bundle_name,VARCHAR,0,0.0,818
4,bundle_type,VARCHAR,0,0.0,6
5,product_category,VARCHAR,0,0.0,4
6,service_class_category,VARCHAR,0,0.0,4
7,price,DOUBLE,0,0.0,288
8,validity,DOUBLE,0,0.0,13
9,data_volume_mb,DOUBLE,0,0.0,184


##### K-Core Filtering

In [17]:

print("Starting Iterative K-Core Filtering algorithm...")

# 1. Create a copy of the data to work on
con.sql("CREATE OR REPLACE TABLE kcore_data AS SELECT * FROM final_model_data;")

iteration = 1
while True:
    # Calculate row count before the start of the cycle
    old_count = con.sql("SELECT COUNT(*) FROM kcore_data").fetchone()[0]
    
    print(f"\n Iteration {iteration}:")
    print(f"   Current Rows: {old_count:,}")

    # Step A: Filter Bundles (Keep only those sold 5 times or more)
    con.sql("""
        CREATE OR REPLACE TABLE kcore_data AS
        SELECT * FROM kcore_data
        WHERE bundle_id IN (
            SELECT bundle_id 
            FROM kcore_data 
            GROUP BY bundle_id 
            HAVING COUNT(*) >= 5
        );
    """)
    
    # Step B: Filter Users (Keep only those with 3 interactions or more)
    con.sql("""
        CREATE OR REPLACE TABLE kcore_data AS
        SELECT * FROM kcore_data
        WHERE msisdn IN (
            SELECT msisdn 
            FROM kcore_data 
            GROUP BY msisdn 
            HAVING COUNT(*) >= 3
        );
    """)

    # Calculate row count after filtering
    new_count = con.sql("SELECT COUNT(*) FROM kcore_data").fetchone()[0]
    deleted_this_round = old_count - new_count
    
    print(f"   Removed {deleted_this_round:,} interactions this round.")

    # Stopping Condition: If no rows were deleted, it means we have reached stability
    if old_count == new_count:
        print("\nEquilibrium reached! No weak data remaining.")
        break
        
    iteration += 1

# ==========================================
# Final Filtering Report
# ==========================================
final_users = con.sql("SELECT COUNT(DISTINCT msisdn) FROM kcore_data").fetchone()[0]
final_items = con.sql("SELECT COUNT(DISTINCT bundle_id) FROM kcore_data").fetchone()[0]

print("\nFinal eSASRec Matrix Summary:")
print(f"Active Users (Core-3): {final_users:,}")
print(f"Strong Bundles (Core-5): {final_items:,}")
print(f"Total Valid Interactions for Training: {new_count:,}")

Starting Iterative K-Core Filtering algorithm...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


 Iteration 1:
   Current Rows: 104,158,526


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   Removed 0 interactions this round.

Equilibrium reached! No weak data remaining.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Final eSASRec Matrix Summary:
Active Users (Core-3): 2,525,144
Strong Bundles (Core-5): 708
Total Valid Interactions for Training: 103,238,669


* Save Processed Data

In [18]:


print("Extracting and saving Golden Checkpoints...")

# ==========================================
# 1. Save Full Filtered Data (K-Core Data)
# ==========================================
# This file contains the human-readable chronological sequence
full_data_path = "kcore_filtered_data.parquet"
con.sql(f"COPY kcore_data TO '{full_data_path}' (FORMAT PARQUET);")
print(f"Full chronological sequence saved to: {full_data_path}")

# ==========================================
# 2. Extract and Save Unique Item Catalog
# ==========================================
# This is the most important table for the future! 
# It contains each bundle only once with its unique attributes.
# We will use it for translating recommendations and post-filtering.
catalog_path = "item_catalog.parquet"
extract_catalog_query = f"""
COPY (
    SELECT DISTINCT 
        bundle_id,
        bundle_name,
        bundle_type,
        product_category,
        service_class_category,
        price,
        validity,
        data_volume_mb,
        voice_volume_min,
        sms_volume_count
    FROM kcore_data
) TO '{catalog_path}' (FORMAT PARQUET);
"""
con.sql(extract_catalog_query)
print(f"Unique Item Catalog extracted and saved to: {catalog_path}")

print("\nData secured successfully!")

Extracting and saving Golden Checkpoints...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Full chronological sequence saved to: kcore_filtered_data.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unique Item Catalog extracted and saved to: item_catalog.parquet

Data secured successfully! We are fully ready for the Tokenization process.


##### Dense Indexing 

In [4]:

print("Reading filtered data from the processed_data folder...")


processed_path = os.path.expanduser('~/processed_data')
kcore_file = f"{processed_path}/kcore_filtered_data.parquet"


con.sql(f"CREATE OR REPLACE TABLE kcore_data AS SELECT * FROM read_parquet('{kcore_file}');")

print("Executing Dense Indexing...")


con.sql("""
CREATE OR REPLACE TABLE user_map AS
SELECT 
    msisdn, 
    ROW_NUMBER() OVER (ORDER BY msisdn) AS user_id
FROM (SELECT DISTINCT msisdn FROM kcore_data);
""")


con.sql("""
CREATE OR REPLACE TABLE item_map AS
SELECT 
    bundle_id, 
    ROW_NUMBER() OVER (ORDER BY bundle_id) AS item_id
FROM (SELECT DISTINCT bundle_id FROM kcore_data);
""")


con.sql("""
CREATE OR REPLACE TABLE tokenized_data AS
SELECT 
    u.user_id,
    i.item_id,
    k.tbl_dt
FROM kcore_data k
JOIN user_map u ON k.msisdn = u.msisdn
JOIN item_map i ON k.bundle_id = i.bundle_id
ORDER BY u.user_id, k.tbl_dt;
""")


con.sql(f"COPY user_map TO '{processed_path}/user_mapping.parquet' (FORMAT PARQUET);")
con.sql(f"COPY item_map TO '{processed_path}/item_mapping.parquet' (FORMAT PARQUET);")
con.sql(f"COPY tokenized_data TO '{processed_path}/tokenized_data.parquet' (FORMAT PARQUET);")


num_users = con.sql("SELECT MAX(user_id) FROM user_map").fetchone()[0]
num_items = con.sql("SELECT MAX(item_id) FROM item_map").fetchone()[0]

print("\nDense Indexing completed successfully and files have been saved!")
print(f"Total tokenized users: {num_users:,}")
print(f"Total tokenized bundles: {num_items:,}")


display(con.sql("SELECT * FROM tokenized_data LIMIT 10").df())

Reading filtered data from the processed_data folder...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Executing Dense Indexing...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Dense Indexing completed successfully and files have been saved!
Total tokenized users: 2,525,144
Total tokenized bundles: 708


,user_id,item_id,tbl_dt
0,1,9,20250814
1,1,151,20250816
2,1,9,20250818
3,1,192,20250820
4,1,191,20250825
5,1,143,20250922
6,1,192,20251029
7,1,193,20251103
8,1,193,20251104
9,1,192,20251110


##### Chronological Sequence Generation

In [ ]:
import os
import glob
import duckdb

print("Activating 'Micro-Chunking' processing protocol (Maximum memory protection)...")

# ==========================================
# 1. Setup paths, environment, and strict memory protection
# ==========================================
processed_path = os.path.expanduser('~/processed_data')
duck_temp = "/tmp/duckdb_spill_micro"
os.makedirs(duck_temp, exist_ok=True)

con = duckdb.connect()
con.sql(f"PRAGMA temp_directory='{duck_temp}';")
con.sql("PRAGMA memory_limit='2GB';")  # Strict limit to force disk spilling
con.sql("PRAGMA threads=1;")           # Single thread to prevent parallel explosion

# ==========================================
# 2. Get the maximum number of users
# ==========================================
max_user_query = f"SELECT MAX(user_id) FROM read_parquet('{processed_path}/tokenized_data.parquet')"
max_user_id = con.sql(max_user_query).fetchone()[0]
print(f"Total number of users to process: {max_user_id:,}")

# Reduce batch size by 90%
chunk_size = 50000 
print(f"Data will be partitioned into micro-chunks of {chunk_size:,} users...\n")

# ==========================================
# 3. Micro-Chunk processing
# ==========================================
for start_id in range(1, int(max_user_id) + 1, chunk_size):
    end_id = start_id + chunk_size - 1
    file_name = f"{processed_path}/user_sequences_part_{start_id}.parquet"
    
    # Skip chunk if already processed (highly useful if the server disconnects and you need to resume)
    if os.path.exists(file_name):
        print(f"Chunk {start_id:,} to {end_id:,} already exists, skipping.")
        continue
        
    print(f"Processing users from {start_id:,} to {end_id:,}...")
    
    chunk_query = f"""
    COPY (
        SELECT 
            user_id,
            LIST(item_id ORDER BY tbl_dt ASC) AS item_sequence,
            LIST(tbl_dt ORDER BY tbl_dt ASC) AS date_sequence
        FROM read_parquet('{processed_path}/tokenized_data.parquet')
        WHERE user_id BETWEEN {start_id} AND {end_id}
        GROUP BY user_id
    ) TO '{file_name}' (FORMAT PARQUET);
    """
    try:
        con.sql(chunk_query)
    except Exception as e:
        print(f"   Error during chunk processing: {e}")
        raise

# ==========================================
# 4. Streaming Merge
# ==========================================
print("\nMicro-partitioning complete! Merging files...")
merge_query = f"""
COPY (
    SELECT * FROM read_parquet('{processed_path}/user_sequences_part_*.parquet')
) TO '{processed_path}/user_sequences.parquet' (FORMAT PARQUET);
"""

try:
    con.sql(merge_query)
    print("Merge successful!")
    
    print(" Cleaning up temporary files...")
    part_files = glob.glob(f"{processed_path}/user_sequences_part_*.parquet")
    for f in part_files:
        os.remove(f)
    print(f"Deleted {len(part_files)} temporary files.")
    
    total_rows = con.sql(f"SELECT COUNT(*) FROM read_parquet('{processed_path}/user_sequences.parquet')").fetchone()[0]
    print(f"\nPipeline successfully completed! The final file contains {total_rows:,} sequences.")
    display(con.sql(f"SELECT * FROM read_parquet('{processed_path}/user_sequences.parquet') LIMIT 5").df())

except Exception as e:
    print(f"An error occurred during the merging phase: {e}")

Activating 'Micro-Chunking' processing protocol (Maximum memory protection)...
Total number of users to process: 2,525,144
Data will be partitioned into micro-chunks of 50,000 users...

Processing users from 1 to 50,000...


Processing users from 50,001 to 100,000...
Processing users from 100,001 to 150,000...
Processing users from 150,001 to 200,000...
Processing users from 200,001 to 250,000...
Processing users from 250,001 to 300,000...
Processing users from 300,001 to 350,000...
Processing users from 350,001 to 400,000...
Processing users from 400,001 to 450,000...
Processing users from 450,001 to 500,000...
Processing users from 500,001 to 550,000...
Processing users from 550,001 to 600,000...
Processing users from 600,001 to 650,000...
Processing users from 650,001 to 700,000...
Processing users from 700,001 to 750,000...
Processing users from 750,001 to 800,000...
Processing users from 800,001 to 850,000...
Processing users from 850,001 to 900,000...
Processing users from 900,001 to 950,000...
Processing users from 950,001 to 1,000,000...
Processing users from 1,000,001 to 1,050,000...
Processing users from 1,050,001 to 1,100,000...
Processing users from 1,100,001 to 1,150,000...
Processing users fr

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Merge successful!
🧹 Cleaning up temporary files...
Deleted 51 temporary files.

Pipeline successfully completed! The final file contains 2,525,144 sequences.


,user_id,item_sequence,date_sequence
0,1,"[9, 151, 9, 192, 191, 143, 192, 193, 193, 192,...","[20250814, 20250816, 20250818, 20250820, 20250..."
1,2,"[630, 618, 630, 639, 605, 588]","[20250810, 20251007, 20251014, 20251021, 20251..."
2,3,"[149, 485, 150, 205, 588, 203, 551, 151, 588, ...","[20250801, 20250803, 20250803, 20250805, 20250..."
3,4,"[495, 569, 9, 495, 495, 588, 639, 588, 588, 58...","[20250801, 20250801, 20250802, 20250805, 20250..."
4,5,"[492, 495, 492, 548, 489, 548, 548, 549, 489, ...","[20250801, 20250808, 20250814, 20250819, 20250..."


In [6]:
import os
import duckdb

print("Starting Quality Assurance Check protocol...\n")

processed_path = os.path.expanduser('~/processed_data')
con = duckdb.connect()

# 1. Check total number of users
total_sequences = con.sql(f"SELECT COUNT(*) FROM read_parquet('{processed_path}/user_sequences.parquet')").fetchone()[0]
expected_users = con.sql(f"SELECT COUNT(DISTINCT user_id) FROM read_parquet('{processed_path}/tokenized_data.parquet')").fetchone()[0]

print(f"Count Check:")
print(f"   - Expected number of users: {expected_users:,}")
print(f"   - Number of generated sequences: {total_sequences:,}")
if total_sequences == expected_users:
    print("   Count test: Passed (No users missing).")
else:
    print("   Count test: Failed!")

# 2. Check length consistency (Item Sequence vs Date Sequence)
length_mismatch_query = f"""
SELECT COUNT(*) 
FROM read_parquet('{processed_path}/user_sequences.parquet')
WHERE len(item_sequence) != len(date_sequence)
"""
mismatched_rows = con.sql(length_mismatch_query).fetchone()[0]

print(f"\nLength Consistency Check:")
if mismatched_rows == 0:
    print("    Length test: Passed (Every item has a corresponding date).")
else:
    print(f"   Length test: Failed! {mismatched_rows} users have a length mismatch.")

# 3. Display quick statistics about sequence length (to understand training data distribution)
stats_query = f"""
SELECT 
    MIN(len(item_sequence)) as min_length,
    MAX(len(item_sequence)) as max_length,
    ROUND(AVG(len(item_sequence)), 2) as avg_length
FROM read_parquet('{processed_path}/user_sequences.parquet')
"""
stats = con.sql(stats_query).fetchone()
print(f"\n Sequence Statistics:")
print(f"   - Shortest sequence: {stats[0]} interactions")
print(f"   - Longest sequence: {stats[1]} interactions")
print(f"   - Average interactions per user: {stats[2]}")

print("\n If all tests passed!, your data is perfect and 100% ready for the next step!")

Starting Quality Assurance Check protocol...

Count Check:
   - Expected number of users: 2,525,144
   - Number of generated sequences: 2,525,144
   Count test: Passed (No users missing).

Length Consistency Check:
    Length test: Passed (Every item has a corresponding date).

 Sequence Statistics:
   - Shortest sequence: 3 interactions (should be 3+ thanks to K-Core filtering)
   - Longest sequence: 873 interactions
   - Average interactions per user: 40.88

 If all tests passed!, your data is perfect and 100% ready for the next step!


In [1]:
import os
import pandas as pd
import numpy as np

print("Starting Chronological Time-Split protocol...")

processed_path = os.path.expanduser('~/processed_data')
file_path = f"{processed_path}/user_sequences.parquet"

# 1. Read data (very light on RAM thanks to its density)
print("Loading time sequences...")
df = pd.read_parquet(file_path)

# 2. Set the cutoff date (November 1, 2025)
# Note: If your date format differs slightly, you can adjust this number
CUTOFF_DATE = 20251101 

def temporal_split(row):
    items = row['item_sequence']
    dates = row['date_sequence']
    
    # Find the cutoff point (the first package purchased in November)
    split_idx = len(dates) # Default: All data is for training if there is no November data
    
    for i, d in enumerate(dates):
        if d >= CUTOFF_DATE:
            split_idx = i
            break
            
    # Split the arrays based on the cutoff point
    train_items = items[:split_idx]
    test_items = items[split_idx:]
    
    return train_items, test_items

print("Splitting sequences into (train and test)... This may take a minute...")

# 3. Apply the custom splitting function
df[['train_sequence', 'test_sequence']] = df.apply(temporal_split, axis=1, result_type='expand')

# 4. Logical filtering (Quality Check)
# The user must have (at least two interactions for training) and (at least one interaction for testing)
print("Excluding users not valid for temporal evaluation...")
valid_users = df[
    (df['train_sequence'].apply(len) >= 2) & 
    (df['test_sequence'].apply(len) >= 1)
].copy()

# 5. Final report
print(f"\nStatistics after splitting:")
print(f"   - Total original users: {len(df):,}")
print(f"   - Users perfectly valid for training and testing: {len(valid_users):,}")
print(f"   - Excluded {len(df) - len(valid_users):,} users (either completely new in November, or disappeared in November)")

# 6. Save the final golden copy for PyTorch
final_path = f"{processed_path}/pytorch_ready_data.parquet"
valid_users[['user_id', 'train_sequence', 'test_sequence']].to_parquet(final_path, index=False)

print(f"\nFinal standard matrices saved in: {final_path}")
print("Shape of the data PyTorch will feed on:")
display(valid_users[['user_id', 'train_sequence', 'test_sequence']].head())

Starting Chronological Time-Split protocol...
Loading time sequences...
Splitting sequences into (train and test)... This may take a minute...
Excluding users not valid for temporal evaluation...

Statistics after splitting:
   - Total original users: 2,525,144
   - Users perfectly valid for training and testing: 2,012,885
   - Excluded 512,259 users (either completely new in November, or disappeared in November)

Final standard matrices saved in: /teamspace/studios/this_studio/processed_data/pytorch_ready_data.parquet
Shape of the data PyTorch will feed on:


,user_id,train_sequence,test_sequence
0,1,"[9, 151, 9, 192, 191, 143, 192]","[193, 193, 192, 192, 192, 486, 151]"
1,2,"[630, 618, 630, 639]","[605, 588]"
2,3,"[149, 485, 150, 205, 588, 203, 551, 151, 588, ...","[151, 588, 205, 152, 205, 144, 151, 206, 151]"
3,4,"[495, 569, 9, 495, 495, 588, 639, 588, 588, 58...",[576]
5,6,"[197, 131]",[129]


#### References


* [eSASRec: Enhancing Transformer-based Recommendations in a
Modular Fashion](https://arxiv.org/pdf/2508.06450v1)